# Module 5 — Neural Networks
**Lectures:** Week 7 (neuron, weights & bias, sigmoid/ReLU, hidden layers, learning rate, backpropagation, MLP)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ieso_lab import load_dataset
from ieso_lab.features import add_peak_label
from ieso_lab.tracking import log_run, show_runs

plt.rcParams["figure.figsize"] = (9, 4)
df = load_dataset().dropna().reset_index(drop=True)
print(df.shape)

## 1. One neuron by hand
$z = w_1x_1 + w_2x_2 + b$ → activation. Slides: hidden ReLU layer → sigmoid output, input (2, 1) → ŷ ≈ 0.88.

In [ ]:
def relu(z): return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))

x = np.array([2, 1])
W_hidden = np.array([[1, 1], [-1, -1]])   # row = one hidden neuron
b_hidden = np.array([-1, 2])
w_out, b_out = np.array([1, -1]), 0

# TODO 15: forward pass: h = relu(W_hidden @ x + b_hidden), y_hat = sigmoid(w_out @ h + b_out)
h = None
y_hat = None
print("hidden:", h, "| output:", round(y_hat, 2))   # expect [2 0] and 0.88

Key insight: an MLP with **no** activation is just stacked linear regressions = one linear regression. ReLU is what lets it bend — e.g. learn the U-shape from raw temperature, no HDD/CDD needed.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
df = df.sort_values("timestamp_utc").reset_index(drop=True)
cut = int(len(df) * 0.8)
tr, te = df.iloc[:cut], df.iloc[cut:]
FEATURES = ["temperature_2m", "hour", "dayofweek", "is_holiday", "demand_lag_24h", "demand_lag_168h"]
y_train, y_test = tr["demand_mw"].values, te["demand_mw"].values

def rmse(a, b): return float(np.sqrt(((np.asarray(a) - np.asarray(b)) ** 2).mean()))

In [ ]:
# TODO 16: scale X (fit on train only) AND scale y (fit a 2nd StandardScaler on y_train.reshape(-1, 1))
x_sc = None
y_sc = None
Xtr, Xte = None, None
ytr = None   # scaled y_train as 1-D

mlp = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", max_iter=300,
                   early_stopping=True, random_state=123)
# TODO 17: fit on scaled data, predict test, UN-scale predictions back to MW (y_sc.inverse_transform)
pred = None
print("MLP RMSE:", rmse(y_test, pred))

In [ ]:
log_run("05", "mlp", {"hidden": (32, 16)}, {"rmse": rmse(y_test, pred)})
show_runs()  # compare against Module 4

## 🔨 Break it: the learning rate
`new weight = old weight − learning rate × gradient`. **Predict** the loss curve for lr = 0.00001, 0.001, 0.5.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)  # expected: we stop early on purpose
for lr_ in [1e-5, 1e-3, 0.5]:
    m = MLPRegressor(hidden_layer_sizes=(32, 16), learning_rate_init=lr_, max_iter=60, random_state=123)
    m.fit(Xtr, ytr)
    plt.plot(m.loss_curve_, label=f"lr={lr_}")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend()

## 🔨 Break it: forget to scale
Fit the same MLP on raw `tr[FEATURES]` and raw `y_train` (MW in the tens of thousands). Record the RMSE and explain what happened.

## Why questions
1. Your hand-computed forward pass: why is h₂ = 0? What did ReLU do?
2. Which learning rate curve was "too small", "good", "too large"? Describe each shape.
3. MLP vs Module 4 linear regression in `show_runs()`: was the extra complexity worth it? (Consider: training time, explainability to a manager.)

---
## 🔧 Git task (do this before you start, and finish after)
```bash
git switch main && git pull
git switch -c module-05-nn          # start of module
# ... work ...
git add notebooks/ experiments/runs.csv
git commit -m "Module 5: neuron by hand, MLP regressor, learning-rate experiment"
git push -u origin module-05-nn
```
Then open a **Pull Request** on GitHub → read your own diff → merge → close the module's Issue.

**Git skill this module:** Team Stage — I'll open a PR on a file you're also editing, so you resolve your first merge conflict.